# BGS r-band luminosity-function fit in log-luminosity space

This notebook reads the DESI DR1 BGS NGC and SGC clustering catalogs, combines them, uses `flux_r_dered` to derive the r-band apparent magnitude `r_dered`, converts it to the literature-style magnitude convention $M_r-5\log_{10}h$, converts those magnitudes to luminosities, and estimates a $1/V_{\max}$-corrected luminosity function over the full redshift range used in the analysis, and fits a single Schechter function to that corrected LF.

DESI Legacy Survey optical fluxes are on the AB system and are stored in nanomaggies. The LF derives the r-band apparent magnitude from `flux_r_dered` using $r_{\rm dered}=22.5-2.5\log_{10}(F_r)$. The absolute magnitude convention is $M_r-5\log_{10}h$, matching the common literature convention for quoted Schechter $M_*$.

The actual fit is performed in log-luminosity space. Starting from the usual Schechter function,

$$
\phi(L)\,dL = \phi_* \left(\frac{L}{L_*}\right)^\alpha \exp\left(-\frac{L}{L_*}\right) d\left(\frac{L}{L_*}\right),
$$

and using $dL = \ln(10) L\,d\log_{10}L$, the LF per dex in luminosity is

$$
\phi(\log_{10}L)
= \ln(10)\,\phi_*\left(\frac{L}{L_*}\right)^{\alpha+1}
\exp\left(-\frac{L}{L_*}\right).
$$

Here $L_*$ is the characteristic luminosity. The parameter $\log_{10}L_*$ gives the location where $L/L_*=1$. We intentionally fit one global LF instead of separate redshift-dependent LFs. The $1/V_{\max}$ correction accounts for the fact that faint galaxies are observable over a smaller redshift volume than bright galaxies.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import astropy.units as u
import astropy.cosmology.units as cu
from astropy.cosmology import Planck18
from astropy.table import Table
from astropy.coordinates import SkyCoord

from scipy.optimize import curve_fit, OptimizeWarning
from scipy.integrate import quad
from scipy.stats import chi2 as chi2_dist

mpl.rcParams.update({
    'font.size': 12,
    'axes.linewidth': 1.1,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': False,
})

COSMO = Planck18
h = COSMO.H0.value / 100.0
print(COSMO)
print('h =', h)

In [ ]:
# Configuration
BGS_DIR = Path('/global/cfs/cdirs/desi/survey/catalogs/DA2/LSS/loa-v1/LSScats/v2.1')
BGS_FILES = [
    BGS_DIR / 'BGS_BRIGHT_full.dat.fits',
]

OUTPUT_DIR = Path('/global/homes/z/zzhang13/DESI/Projection/catalogs')
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path.cwd()

Z_FIT_RANGE = (0.1, 0.4)
R_MAG_LIMIT = 19.5   # BGS Bright limit for BGS_BRIGHT_full.dat.fits.
N_LOGL_BINS = 24
LOG_L_MIN_FIT = 9.0

# Correct dereddened r-band flux column in nanomaggies.
FLUX_R_COL = 'flux_r_dered'

# For luminosity in solar units. The exact value depends mildly on r-band definition.
# For LF shape/completeness ratios, changing this constant shifts logL and logL* together.
M_SUN_R_AB = 4.64

# Effective survey area. This affects the LF normalization but not the fitted shape.
# Leave as None to use f_sky=1.0 for shape diagnostics.
SURVEY_AREA_DEG2 = None
F_SKY = 1.0 if SURVEY_AREA_DEG2 is None else SURVEY_AREA_DEG2 / 41252.96124941927

# Optional safety limiter for quick tests. Set to None for the full BGS catalog.
MAX_ROWS_PER_FILE = None

# Deterministic downsampling for fast LF diagnostics.
# Use 1000 to keep every 1000th galaxy; set to None or 1 for the full catalog.
BGS_ROW_STRIDE = 1000

print('BGS files:')
for fn in BGS_FILES:
    print('  ', fn)
print('Output dir:', OUTPUT_DIR)
print('Global LF fit redshift range:', Z_FIT_RANGE)
print('R magnitude limit:', R_MAG_LIMIT)
print('Minimum logL used in LF fit:', LOG_L_MIN_FIT)
print('f_sky used for Vmax:', F_SKY)

## Load the DR2 BGS Bright catalog and assign Galactic cap

The DR2/DA2 BGS Bright sample is stored as a single full-footprint catalog rather than separate NGC and SGC files. We therefore read `BGS_BRIGHT_full.dat.fits` directly and assign a diagnostic Galactic-cap label from sky position. This `CAP` label is used only for LF systematics checks; it is not an input selection flag.

The DR2 column `PROB_OBS` is the probability that a galaxy was observed. We convert it to the BGS/completeness incompleteness weight

$$
w_{\rm completeness}=\frac{1}{P_{\rm obs}}.
$$

This weight corrects the luminosity-function estimator for spectroscopic observation probability. It does not include the cluster geometric fraction, which is computed separately from random catalogs and applied later in the matched cluster catalog.


In [ ]:
def find_column_local(table, candidates):
    for col in candidates:
        if col in table.colnames:
            return col
    return None


def assign_galactic_cap(ra_deg, dec_deg):
    """Assign NGC/SGC from sky position using Galactic latitude."""
    coord = SkyCoord(ra=np.asarray(ra_deg, dtype=float) * u.deg,
                     dec=np.asarray(dec_deg, dtype=float) * u.deg)
    b_gal = coord.galactic.b.deg
    cap = np.where(b_gal >= 0.0, 'NGC', 'SGC')
    cap[~np.isfinite(b_gal)] = 'UNKNOWN'
    return cap


bgs = Table.read(BGS_FILES[0])
if MAX_ROWS_PER_FILE is not None:
    bgs = bgs[:MAX_ROWS_PER_FILE]

if BGS_ROW_STRIDE is not None and BGS_ROW_STRIDE > 1:
    n_before_stride = len(bgs)
    bgs = bgs[::BGS_ROW_STRIDE]
    print(f'Downsampled BGS rows by stride {BGS_ROW_STRIDE}: {n_before_stride:,} -> {len(bgs):,}')

ra_col = find_column_local(bgs, ['RA', 'RA_BGS', 'TARGET_RA'])
dec_col = find_column_local(bgs, ['DEC', 'DEC_BGS', 'TARGET_DEC'])
if ra_col is None or dec_col is None:
    raise KeyError('Could not find RA/DEC columns needed to assign Galactic cap.')

bgs['CAP'] = assign_galactic_cap(bgs[ra_col], bgs[dec_col])

print(f'BGS rows: {len(bgs):,} from {BGS_FILES[0].name}')
print('Galactic cap counts:')
for cap in ['NGC', 'SGC', 'UNKNOWN']:
    print(f'  {cap}: {np.count_nonzero(np.asarray(bgs["CAP"]) == cap):,}')
print('Columns:')
print(bgs.colnames)


## Magnitude convention, luminosity variable, and \(V_{\max}\)

We derive the apparent magnitude from `flux_r_dered` as

$$
r_{\rm dered}=22.5-2.5\log_{10}(F_r),
$$

with $F_r$ in nanomaggies. To match the common literature convention, we work with

$$
M_{r,h}\equiv M_r-5\log_{10}h
= r_{\rm dered} - {\rm DM}(z) - 5\log_{10}h.
$$

This means the fitted magnitude parameter can be compared directly to literature values quoted as $M_*-5\log_{10}h$. We define a corresponding log-luminosity variable by

$$
\log_{10}L_h = -0.4\left(M_{r,h}-M_{\odot,r}\right),
$$

where the solar constant only sets the horizontal zero point. The LF shape and completeness ratios do not depend on this arbitrary zero point. We fit only galaxies with $\log_{10}L_h>9$; below this scale a double-Schechter form may be required.

The apparent magnitude limit implies a maximum observable redshift for each galaxy:

$$
r_{\rm lim} = M_{r,h,i} + {\rm DM}(z_{\max,i}) + 5\log_{10}h.
$$

Rather than solving this equation separately for every galaxy, we precompute a redshift grid for

$$
q(z)={\rm DM}(z)+5\log_{10}h
$$

and use interpolation to invert

$$
q(z_{\max,i})=r_{\rm lim}-M_{r,h,i}.
$$

The accessible comoving volume is

$$
V_{\max,i}
= f_{\rm sky}\left[V_c(z_{\max,i}^{\rm eff})-V_c(z_{\min})\right],
$$

where $z_{\max,i}^{\rm eff}=\min(z_{\max,i},z_{\rm high})$. Astropy returns volumes in ${\rm Mpc}^3$; below we multiply by $h^3$ so the volume is in $(h^{-1}{\rm Mpc})^3$ and the LF normalization is in $h^3{\rm Mpc}^{-3}{\rm dex}^{-1}$.

In [ ]:
def find_column(table, candidates):
    for col in candidates:
        if col in table.colnames:
            return col
    return None


def flux_to_mag_nanomaggy(flux):
    flux = np.asarray(flux, dtype=float)
    mag = np.full(len(flux), np.nan, dtype=float)
    good = np.isfinite(flux) & (flux > 0)
    mag[good] = 22.5 - 2.5 * np.log10(flux[good])
    return mag


def distance_modulus(z):
    z = np.asarray(z, dtype=float)
    d_l = (z * cu.redshift).to(u.pc, cu.redshift_distance(COSMO, kind='luminosity'))
    return 5.0 * np.log10(d_l.value / 10.0)


def M_minus_5logh_from_apparent(mag, z):
    """Return M - 5 log10(h), matching common LF literature convention."""
    return np.asarray(mag, dtype=float) - distance_modulus(z) - 5.0 * np.log10(h)


def apparent_mag_from_M_minus_5logh(M_h, z):
    return np.asarray(M_h, dtype=float) + distance_modulus(z) + 5.0 * np.log10(h)


def logL_from_M_minus_5logh(M_h, M_sun=M_SUN_R_AB):
    """Log-luminosity variable based on M - 5 log10(h)."""
    return -0.4 * (np.asarray(M_h, dtype=float) - M_sun)


def magnitude_limit_to_logL(z, m_lim=R_MAG_LIMIT):
    M_lim_h = M_minus_5logh_from_apparent(m_lim, z)
    return logL_from_M_minus_5logh(M_lim_h)


def comoving_volume_hmpc3(z):
    """Full-sky comoving volume in (h^-1 Mpc)^3."""
    return COSMO.comoving_volume(z).to_value(u.Mpc**3) * h**3


def compute_vmax_hmpc3_grid(M_h, z_low, z_high, f_sky=F_SKY, m_lim=R_MAG_LIMIT, n_grid=20000):
    """Fast vectorized Vmax calculation using a precomputed redshift grid."""
    M_h = np.asarray(M_h, dtype=float)
    z_grid = np.linspace(z_low, z_high, n_grid)
    q_grid = distance_modulus(z_grid) + 5.0 * np.log10(h)
    target = m_lim - M_h

    q_low = q_grid[0]
    q_high = q_grid[-1]
    zmax = np.full(len(M_h), np.nan, dtype=float)

    visible_to_high = np.isfinite(target) & (target >= q_high)
    visible_somewhere = np.isfinite(target) & (target >= q_low) & (target < q_high)
    zmax[visible_to_high] = z_high
    zmax[visible_somewhere] = np.interp(target[visible_somewhere], q_grid, z_grid)

    zmax_eff = np.minimum(zmax, z_high)
    v_grid = comoving_volume_hmpc3(z_grid)
    v_at_zmax = np.full(len(M_h), np.nan, dtype=float)
    good = np.isfinite(zmax_eff)
    v_at_zmax[good] = np.interp(zmax_eff[good], z_grid, v_grid)

    v_low = v_grid[0]
    vmax = f_sky * (v_at_zmax - v_low)
    vmax[~np.isfinite(vmax) | (vmax <= 0)] = np.nan
    return vmax, zmax_eff

z_col = find_column(bgs, ['Z', 'Z_BGS', 'z'])
if z_col is None:
    raise KeyError('Could not find a redshift column.')

flux_r_col = FLUX_R_COL
if flux_r_col not in bgs.colnames:
    raise KeyError(f'Expected dereddened r-band flux column {flux_r_col!r}, but it is not in the BGS table.')
mag_r = flux_to_mag_nanomaggy(bgs[flux_r_col])
print(f'Derived r_dered from dereddened r-band flux column: {flux_r_col}')

prob_obs_col = find_column(bgs, ['PROB_OBS', 'prob_obs'])
if prob_obs_col is None:
    raise KeyError('DR2 BGS catalog is expected to contain PROB_OBS, but it was not found.')
prob_obs = np.asarray(bgs[prob_obs_col], dtype=float)
with np.errstate(divide='ignore', invalid='ignore'):
    comp_weight = 1.0 / prob_obs
comp_weight[~np.isfinite(comp_weight) | (comp_weight <= 0)] = np.nan
bgs['COMP_WEIGHT'] = comp_weight
print(f'Using {prob_obs_col} as observation probability; COMP_WEIGHT = 1 / PROB_OBS')
print('PROB_OBS percentiles:', np.nanpercentile(prob_obs[np.isfinite(prob_obs)], [0, 1, 16, 50, 84, 99, 100]))
print('COMP_WEIGHT percentiles:', np.nanpercentile(comp_weight[np.isfinite(comp_weight)], [0, 1, 16, 50, 84, 99, 100]))

z = np.asarray(bgs[z_col], dtype=float)
M_r_h = M_minus_5logh_from_apparent(mag_r, z)
logL = logL_from_M_minus_5logh(M_r_h)

zlo_fit, zhi_fit = Z_FIT_RANGE
base_pre_vmax = (
    np.isfinite(z)
    & np.isfinite(mag_r)
    & np.isfinite(M_r_h)
    & np.isfinite(logL)
    & (z >= zlo_fit)
    & (z < zhi_fit)
    & (mag_r < R_MAG_LIMIT)
    & (logL > LOG_L_MIN_FIT)
    & np.isfinite(comp_weight)
    & (comp_weight > 0)
)

vmax_hmpc3 = np.full(len(bgs), np.nan, dtype=float)
zmax_eff = np.full(len(bgs), np.nan, dtype=float)
vmax_hmpc3[base_pre_vmax], zmax_eff[base_pre_vmax] = compute_vmax_hmpc3_grid(
    M_r_h[base_pre_vmax],
    zlo_fit,
    zhi_fit,
    f_sky=F_SKY,
    m_lim=R_MAG_LIMIT,
)

base = base_pre_vmax & np.isfinite(vmax_hmpc3) & (vmax_hmpc3 > 0)

print('z column:', z_col)
print(f'Rows passing finite + derived r_dered<{R_MAG_LIMIT} + logL>{LOG_L_MIN_FIT} + finite PROB_OBS: {np.count_nonzero(base):,}')
print(f'M_r - 5log10(h) range: {np.nanpercentile(M_r_h[base], [1, 50, 99])}')
print(f'log10(L_h) range: {np.nanpercentile(logL[base], [1, 50, 99])}')
print(f'Vmax [(h^-1 Mpc)^3] range: {np.nanpercentile(vmax_hmpc3[base], [1, 50, 99])}')

## Schechter model in log-luminosity space

We estimate the binned LF with a weighted $1/V_{\max}$ estimator. For DR2 BGS Bright we use

$$
w_i = w_{{\rm completeness},i}=\frac{1}{P_{{\rm obs},i}},
$$

where `PROB_OBS` is the observation probability supplied by the DR2 catalog. The geometric aperture correction is not included here; it is a cluster-level correction derived from random catalogs and applied later to the matched cluster catalog.

The weighted luminosity function in log-luminosity bin $j$ is

$$
\phi_j = \frac{1}{\Delta\log_{10}L_j}
\sum_{i\in j}\frac{w_i}{V_{\max,i}}.
$$

The vertical uncertainty is the weighted Poisson uncertainty,

$$
\sigma^2_{\phi,j}=\frac{1}{(\Delta\log_{10}L_j)^2}
\sum_{i\in j}\left(\frac{w_i}{V_{\max,i}}\right)^2.
$$

The horizontal uncertainty is represented by the half-width of each log-luminosity bin,

$$
\sigma_{x,j}=\frac{1}{2}\Delta\log_{10}L_j.
$$

For fitting, we optionally propagate this horizontal uncertainty into the vertical direction using the local model derivative,

$$
\sigma^2_{{\rm eff},j}
= \sigma^2_{\phi,j}
+ \left(\frac{d\phi}{d\log_{10}L}\bigg|_j\sigma_{x,j}\right)^2,
$$

ignoring any covariance between x and y errors. The model is

$$
\phi(\log_{10}L)=\ln(10)\phi_* x^{\alpha+1}e^{-x},
\qquad
x=10^{\log_{10}L-\log_{10}L_*}.
$$

The fitted normalization has the units set by the $V_{\max}$ convention, while the shape parameters are $\log_{10}L_*$ and $\alpha$. Parameter errors are estimated from the fit covariance matrix returned by `curve_fit`; the 1-sigma errors are $\sigma_p=\sqrt{\mathrm{diag}(\mathbf{C}_p)}$, with $\sigma_{M_*}=2.5\,\sigma_{\log_{10}L_*}$.


In [ ]:
def schechter_logL(logL_values, log_phi_star, logL_star, alpha):
    phi_star = np.exp(log_phi_star)
    x = 10.0 ** (np.asarray(logL_values, dtype=float) - logL_star)
    return np.log(10.0) * phi_star * x ** (alpha + 1.0) * np.exp(-x)


def schechter_logL_shape(logL_values, logL_star, alpha):
    x = 10.0 ** (np.asarray(logL_values, dtype=float) - logL_star)
    return np.log(10.0) * x ** (alpha + 1.0) * np.exp(-x)


def schechter_logL_derivative(logL_values, log_phi_star, logL_star, alpha):
    """Derivative d phi / d log10(L)."""
    ell = np.asarray(logL_values, dtype=float)
    phi = schechter_logL(ell, log_phi_star, logL_star, alpha)
    x = 10.0 ** (ell - logL_star)
    return np.log(10.0) * phi * ((alpha + 1.0) - x)


def effective_yerr_from_xyerr(x, yerr, xerr, params):
    """Propagate x-errors into y-errors using the local model slope."""
    log_phi_star, logL_star, alpha = params
    dydx = schechter_logL_derivative(x, log_phi_star, logL_star, alpha)
    return np.sqrt(yerr**2 + (dydx * xerr)**2)


def fit_schechter_logL_vmax(logL_values, vmax_values, weights=None, n_bins=N_LOGL_BINS, include_xerr=True):
    logL_values = np.asarray(logL_values, dtype=float)
    vmax_values = np.asarray(vmax_values, dtype=float)
    if weights is None:
        weights = np.ones(len(logL_values), dtype=float)
    weights = np.asarray(weights, dtype=float)

    good_input = (
        np.isfinite(logL_values)
        & np.isfinite(vmax_values)
        & (vmax_values > 0)
        & np.isfinite(weights)
        & (weights > 0)
    )
    logL_values = logL_values[good_input]
    vmax_values = vmax_values[good_input]
    weights = weights[good_input]
    if len(logL_values) < 50:
        raise ValueError('Need at least 50 galaxies for a stable diagnostic fit')

    lo, hi = np.nanpercentile(logL_values, [1, 99])
    edges = np.linspace(lo, hi, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)
    xerr = 0.5 * widths

    weighted_inv_vmax = weights / vmax_values
    bin_index = np.digitize(logL_values, edges) - 1

    phi = np.zeros(n_bins, dtype=float)
    phi_var = np.zeros(n_bins, dtype=float)
    counts = np.zeros(n_bins, dtype=int)
    weight_sum = np.zeros(n_bins, dtype=float)

    for ibin in range(n_bins):
        in_bin = bin_index == ibin
        counts[ibin] = np.count_nonzero(in_bin)
        if counts[ibin] == 0:
            continue
        weight_sum[ibin] = np.sum(weights[in_bin])
        phi[ibin] = np.sum(weighted_inv_vmax[in_bin]) / widths[ibin]
        phi_var[ibin] = np.sum(weighted_inv_vmax[in_bin] ** 2) / widths[ibin] ** 2

    phi_err = np.sqrt(phi_var)
    good = (counts > 0) & np.isfinite(phi) & np.isfinite(phi_err) & (phi > 0) & (phi_err > 0)
    centers_fit = centers[good]
    phi_fit = phi[good]
    phi_err_fit = phi_err[good]
    xerr_fit = xerr[good]

    p0 = [np.log(np.nanmax(phi_fit)), centers_fit[np.nanargmax(phi_fit)], -1.0]
    bounds = ([-80.0, 6.0, -2.5], [20.0, 13.0, 0.5])

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', OptimizeWarning)
        popt_yonly, _ = curve_fit(
            schechter_logL,
            centers_fit,
            phi_fit,
            p0=p0,
            sigma=phi_err_fit,
            absolute_sigma=True,
            bounds=bounds,
            maxfev=50000,
        )

        if include_xerr:
            phi_err_eff = effective_yerr_from_xyerr(
                centers_fit,
                phi_err_fit,
                xerr_fit,
                popt_yonly,
            )
        else:
            phi_err_eff = phi_err_fit

        popt, pcov = curve_fit(
            schechter_logL,
            centers_fit,
            phi_fit,
            p0=popt_yonly,
            sigma=phi_err_eff,
            absolute_sigma=True,
            bounds=bounds,
            maxfev=50000,
        )

    model_fit = schechter_logL(centers_fit, *popt)
    chi2 = np.sum(((phi_fit - model_fit) / phi_err_eff) ** 2)
    dof = len(phi_fit) - len(popt)

    return {
        'logL': logL_values,
        'vmax': vmax_values,
        'weights': weights,
        'edges': edges,
        'centers': centers,
        'widths': widths,
        'xerr': xerr,
        'counts': counts,
        'weight_sum': weight_sum,
        'phi': phi,
        'phi_err': phi_err,
        'density_centers': centers_fit,
        'density': phi_fit,
        'density_err': phi_err_fit,
        'density_xerr': xerr_fit,
        'density_err_eff': phi_err_eff,
        'log_phi_star': popt[0],
        'logL_star': popt[1],
        'alpha': popt[2],
        'cov': pcov,
        'chi2': chi2,
        'dof': dof,
        'N': len(logL_values),
        'weight_sum_total': np.sum(weights),
        'median_weight': np.nanmedian(weights),
    }


def M_minus_5logh_from_logL(logL_value, M_sun=M_SUN_R_AB):
    return M_sun - 2.5 * np.asarray(logL_value)

def parameter_errors_from_covariance(pcov):
    """Return 1-sigma parameter errors from the fit covariance matrix."""
    pcov = np.asarray(pcov, dtype=float)
    if pcov.shape != (3, 3):
        return np.full(3, np.nan)
    diag = np.diag(pcov)
    err = np.full(3, np.nan, dtype=float)
    good = np.isfinite(diag) & (diag >= 0)
    err[good] = np.sqrt(diag[good])
    return err


In [ ]:
zlo_fit, zhi_fit = Z_FIT_RANGE
fit_mask_global = base
global_fit = fit_schechter_logL_vmax(
    logL[fit_mask_global],
    vmax_hmpc3[fit_mask_global],
    weights=comp_weight[fit_mask_global],
)
global_fit['idx'] = np.where(fit_mask_global)[0]
global_fit['zlo'] = zlo_fit
global_fit['zhi'] = zhi_fit
global_fit['M_star_minus_5logh'] = M_minus_5logh_from_logL(global_fit['logL_star'])
param_err = parameter_errors_from_covariance(global_fit['cov'])
global_fit['log_phi_star_err'] = param_err[0]
global_fit['logL_star_err'] = param_err[1]
global_fit['alpha_err'] = param_err[2]
global_fit['M_star_minus_5logh_err'] = 2.5 * global_fit['logL_star_err']

# Goodness-of-fit diagnostics on the binned LF used in the fit.
y = np.asarray(global_fit['density'], dtype=float)
yerr = np.asarray(global_fit['density_err_eff'], dtype=float)
xfit = np.asarray(global_fit['density_centers'], dtype=float)
ymod = schechter_logL(
    xfit,
    global_fit['log_phi_star'],
    global_fit['logL_star'],
    global_fit['alpha'],
)
resid = y - ymod
pull = resid / yerr
resid_dex = np.log10(y) - np.log10(ymod)
resid_dex_err = yerr / (y * np.log(10.0))
chi2 = np.sum(pull**2)
dof = len(y) - 3
p_value = chi2_dist.sf(chi2, dof)

# Gaussian-error information criteria up to an additive constant.
loglike = -0.5 * np.sum((resid / yerr)**2 + np.log(2.0 * np.pi * yerr**2))
npar = 3
ndata = len(y)
aic = 2 * npar - 2 * loglike
bic = npar * np.log(ndata) - 2 * loglike

summary = pd.DataFrame([
    {
        'z_low': global_fit['zlo'],
        'z_high': global_fit['zhi'],
        'N_galaxies': global_fit['N'],
        'weighted_N_galaxies': global_fit['weight_sum_total'],
        'median_COMP_WEIGHT': global_fit['median_weight'],
        'N_fit_bins': ndata,
        'log10_L_star': global_fit['logL_star'],
        'log10_L_star_err': global_fit['logL_star_err'],
        'M_star_minus_5logh': global_fit['M_star_minus_5logh'],
        'M_star_minus_5logh_err': global_fit['M_star_minus_5logh_err'],
        'alpha': global_fit['alpha'],
        'alpha_err': global_fit['alpha_err'],
        'log_phi_star': global_fit['log_phi_star'],
        'log_phi_star_err': global_fit['log_phi_star_err'],
        'phi_star': np.exp(global_fit['log_phi_star']),
        'phi_star_err': np.exp(global_fit['log_phi_star']) * global_fit['log_phi_star_err'],
        'chi2': chi2,
        'dof': dof,
        'reduced_chi2': chi2 / dof if dof > 0 else np.nan,
        'p_value': p_value,
        'aic': aic,
        'bic': bic,
        'residual_mean': np.mean(resid),
        'residual_std': np.std(resid, ddof=1),
        'pull_mean': np.mean(pull),
        'pull_std': np.std(pull, ddof=1),
        'residual_dex_mean': np.mean(resid_dex),
        'residual_dex_std': np.std(resid_dex, ddof=1),
        'median_residual_dex_err': np.nanmedian(resid_dex_err),
        'median_xerr_dex': np.nanmedian(global_fit['density_xerr']),
        'median_yerr_raw': np.nanmedian(global_fit['density_err']),
        'median_yerr_eff': np.nanmedian(global_fit['density_err_eff']),
        'f_sky': F_SKY,
        'r_mag_limit': R_MAG_LIMIT,
        'logL_min_fit': LOG_L_MIN_FIT,
    }
])
display(summary)

## Literature comparison values

Published LF parameters are often quoted as $M_* - 5\log_{10}h$. The notebook now uses this same convention directly, so no additional h-conversion is needed for the magnitude comparison. The corresponding horizontal plotting coordinate is defined as

$$
\log_{10}L_{*,h}=-0.4\left[(M_*-5\log_{10}h)-M_{\odot,r}\right].
$$

In [ ]:
literature = pd.DataFrame([
    {
        'label': 'SDSS Blanton+03 0.1r',
        'M_star_minus_5logh': -20.44,
        'alpha': -1.05,
    },
    {
        'label': 'SDSS Blanton+01 r*',
        'M_star_minus_5logh': -20.83,
        'alpha': -1.20,
    },
])
literature['log10_L_star'] = logL_from_M_minus_5logh(literature['M_star_minus_5logh'].values)
display(literature)

## Global \(1/V_{\max}\)-corrected LF fit for all BGS galaxies in \(0.1<z<0.4\)

This plot uses all BGS galaxies in the global redshift range. The black points show the $1/V_{\max}$-corrected LF estimate in bins of \(\log_{10}L\), and the red curve is the best-fit single Schechter function. The dashed red line marks the fitted \(\log_{10}L_*\). The lower panel shows normalized residuals, \((\phi_{\rm data}-\phi_{\rm model})/\sigma\).

In [ ]:
fig, (ax, rax) = plt.subplots(
    2, 1,
    figsize=(7.0, 6.2),
    sharex=True,
    gridspec_kw={'height_ratios': [3.0, 1.0], 'hspace': 0.05},
)

xgrid = np.linspace(np.nanmin(global_fit['density_centers']) - 0.1, np.nanmax(global_fit['density_centers']) + 0.1, 600)
ygrid = schechter_logL(xgrid, global_fit['log_phi_star'], global_fit['logL_star'], global_fit['alpha'])

ax.errorbar(
    global_fit['density_centers'],
    global_fit['density'],
    xerr=global_fit['density_xerr'],
    yerr=global_fit['density_err_eff'],
    fmt='o',
    ms=4,
    color='black',
    ecolor='0.35',
    capsize=0,
    label=rf"$1/V_{{\max}}$ BGS, ${global_fit['zlo']:.1f}<z<{global_fit['zhi']:.1f}$, $N={global_fit['N']:,}$",
)
ax.plot(xgrid, ygrid, color='tab:red', lw=2.3, label=r'$1/V_{\max}$ single Schechter fit')
ax.axvline(LOG_L_MIN_FIT, color='0.35', ls='-.', lw=1.3, label=rf'$\log_{{10}}L_h>{LOG_L_MIN_FIT:.1f}$ cut')
ax.axvline(
    global_fit['logL_star'],
    color='tab:red',
    ls='--',
    lw=1.7,
    label=rf"$\log_{{10}}L_*={global_fit['logL_star']:.2f}$",
)

for _, row in literature.iterrows():
    ax.axvline(row['log10_L_star'], color='0.65', ls=':', lw=1.3, alpha=0.8)

ax.set_yscale('log')
ax.set_ylabel(r'$\phi(\log_{10}L)$ [$h^3\,{\rm Mpc}^{-3}{\rm dex}^{-1}$]')
ax.legend(loc='upper left', fontsize=9)
ax.text(
    0.97,
    0.05,
    rf"$M_*-5\log h={global_fit['M_star_minus_5logh']:.2f}\pm{global_fit['M_star_minus_5logh_err']:.2f}$" + '\n' +
    rf"$\alpha={global_fit['alpha']:.2f}\pm{global_fit['alpha_err']:.2f}$" + '\n' +
    rf"$\chi^2_\nu={summary.loc[0, 'reduced_chi2']:.2f}$" + '\n' +
    rf"$p={summary.loc[0, 'p_value']:.2g}$",
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    bbox=dict(facecolor='white', edgecolor='none', alpha=0.85),
)

rax.errorbar(xfit, resid_dex, xerr=global_fit['density_xerr'], yerr=resid_dex_err, fmt='o', ms=4, color='black', ecolor='0.5', capsize=0)
rax.set_xlabel(r'$\log_{10}L_h$')
rax.set_ylabel(r'$\Delta\log_{10}\phi$ [dex]')
rax.axhline(0.0, color='black', lw=1.0)

for axis in (ax, rax):
    axis.tick_params(direction='in', top=True, right=True)
    for spine in axis.spines.values():
        spine.set_linewidth(1.1)

fig.tight_layout()
plt.show()

## Global Schechter shape as a function of $L/L_*$

This view removes the fitted luminosity scale and shows how the global faint-end slope and exponential cutoff shape the function.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.8))

x = np.logspace(-2, 2, 500)  # L/L*
shape = np.log(10.0) * x ** (global_fit['alpha'] + 1.0) * np.exp(-x)
shape /= np.nanmax(shape)
ax.plot(x, shape, color='tab:red', lw=2.4,
        label=rf"global fit, $\alpha={global_fit['alpha']:.2f}$")

ax.axvline(1.0, color='black', ls='--', lw=1.4, label=r'$L=L_*$')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$L/L_*$')
ax.set_ylabel(r'normalized $\phi(\log_{10}L)$')
ax.legend(loc='upper right')
fig.tight_layout()
plt.show()

## Completeness ratio implied by the global LF

The apparent magnitude limit implies a redshift-dependent luminosity limit. The observable fraction relative to a reference luminosity limit is

$$
f_{\rm obs}(z)=
\frac{\int_{\log L_{\rm lim}(z)}^{\infty}\phi(\log L)\,d\log L}
{\int_{\log L_{\rm ref}}^{\infty}\phi(\log L)\,d\log L}.
$$

Then the multiplicative luminosity-function correction is

$$
w_{\rm LF}(z)=\frac{1}{f_{\rm obs}(z)}.
$$

Because we are now using a single global LF, the redshift dependence in $w_{\rm LF}$ comes only from the changing luminosity limit, not from redshift-dependent Schechter parameters.

In [ ]:
def integrate_schechter_logL_from(logL_min, logL_star, alpha, logL_max=14.0):
    value, _ = quad(
        lambda ell: schechter_logL_shape(ell, logL_star, alpha),
        logL_min,
        logL_max,
        epsabs=1e-10,
        epsrel=1e-6,
        limit=200,
    )
    return value


def lf_observed_fraction_logL(logL_lim, logL_ref, logL_star, alpha):
    denom = integrate_schechter_logL_from(logL_ref, logL_star, alpha)
    logL_lim_eff = np.maximum(np.asarray(logL_lim, dtype=float), logL_ref)
    numer = np.array([
        integrate_schechter_logL_from(ell, logL_star, alpha)
        for ell in np.atleast_1d(logL_lim_eff)
    ])
    return np.clip(numer / denom, 0.0, 1.0)

logL_ref_raw = magnitude_limit_to_logL(0.1, R_MAG_LIMIT)
logL_ref = max(logL_ref_raw, LOG_L_MIN_FIT)
z_grid = np.linspace(0.1, 0.4, 200)
logL_lim_grid = magnitude_limit_to_logL(z_grid, R_MAG_LIMIT)

frac = lf_observed_fraction_logL(
    logL_lim_grid,
    logL_ref,
    global_fit['logL_star'],
    global_fit['alpha'],
)
weight = 1.0 / frac

fig, ax = plt.subplots(figsize=(6.5, 4.8))
ax.plot(z_grid, weight, color='tab:red', lw=2.4, label='global LF correction')
ax.axhline(1.0, color='black', lw=1.2, ls='--')
ax.axvspan(zlo_fit, zhi_fit, color='0.8', alpha=0.12)
ax.set_xlabel(r'$z$')
ax.set_ylabel(r'$w_{\rm LF}=1/f_{\rm obs}$')
ax.set_ylim(bottom=0.8)
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()

M_ref = M_minus_5logh_from_apparent(R_MAG_LIMIT, 0.1)
print(f'M_ref for h-scaled r<{R_MAG_LIMIT} at z=0.1 is {M_ref:.3f}')
print(f'raw log10(L_ref_h) for h-scaled r<{R_MAG_LIMIT} at z=0.1 is {logL_ref_raw:.3f}')
print(f'effective log10(L_ref_h) used after logL cut is {logL_ref:.3f}')
print(f'global log10(L*_h) = {global_fit["logL_star"]:.3f}')
print(f'global M* - 5log10(h) = {global_fit["M_star_minus_5logh"]:.3f}')

## Systematics diagnostics for LF residual trends

The cells below test whether the systematic residual trend is driven by sample definition or selection effects. Each diagnostic repeats the same weighted $1/V_{\max}$ LF fit machinery but changes the subsample:

- residuals colored by mean redshift in each luminosity bin,
- NGC versus SGC, where cap is assigned from Galactic latitude,
- weighted versus unweighted LF fits, to show the impact of `PROB_OBS`,
- red versus blue galaxies,
- brighter apparent-magnitude cuts within the BGS Bright catalog,
- narrow redshift range $0.1<z<0.2$.

The red/blue split is intentionally configurable because the best color boundary depends on whether the color is rest-frame/K-corrected. Here it is a first diagnostic, not a final population decomposition.


In [ ]:
# Diagnostic configuration.
BGS_BRIGHT_LIMIT = 19.5
BRIGHTER_LIMIT = 19.0
COLOR_SPLIT_GR = 0.7
N_DIAG_BINS = N_LOGL_BINS

# Build a simple observed-frame dereddened g-r color if g flux is available.
if 'flux_g_dered' in bgs.colnames:
    mag_g = flux_to_mag_nanomaggy(bgs['flux_g_dered'])
elif 'FLUX_G' in bgs.colnames:
    mag_g = flux_to_mag_nanomaggy(bgs['FLUX_G'])
else:
    mag_g = np.full(len(bgs), np.nan)

g_minus_r = mag_g - mag_r
print('finite g-r:', np.count_nonzero(np.isfinite(g_minus_r)))
print('median g-r in LF sample:', np.nanmedian(g_minus_r[base]))

In [ ]:
def make_lf_mask(z_range=Z_FIT_RANGE, mag_limit=R_MAG_LIMIT, extra_mask=None, logL_min=LOG_L_MIN_FIT):
    zlo, zhi = z_range
    mask = (
        np.isfinite(z)
        & np.isfinite(mag_r)
        & np.isfinite(M_r_h)
        & np.isfinite(logL)
        & (z >= zlo)
        & (z < zhi)
        & (mag_r < mag_limit)
        & (logL > logL_min)
        & np.isfinite(comp_weight)
        & (comp_weight > 0)
    )
    if extra_mask is not None:
        mask &= np.asarray(extra_mask, dtype=bool)
    return mask


def compute_vmax_for_mask(mask, z_range=Z_FIT_RANGE, mag_limit=R_MAG_LIMIT):
    zlo, zhi = z_range
    vmax = np.full(np.count_nonzero(mask), np.nan, dtype=float)
    zmax = np.full(np.count_nonzero(mask), np.nan, dtype=float)
    if len(vmax) == 0:
        return vmax, zmax
    vmax, zmax = compute_vmax_hmpc3_grid(
        M_r_h[mask],
        zlo,
        zhi,
        f_sky=F_SKY,
        m_lim=mag_limit,
    )
    return vmax, zmax


def fit_lf_subset(label, z_range=Z_FIT_RANGE, mag_limit=R_MAG_LIMIT, extra_mask=None, logL_min=LOG_L_MIN_FIT, use_comp_weight=True):
    mask0 = make_lf_mask(z_range=z_range, mag_limit=mag_limit, extra_mask=extra_mask, logL_min=logL_min)
    vmax_subset, zmax_subset = compute_vmax_for_mask(mask0, z_range=z_range, mag_limit=mag_limit)
    good = np.isfinite(vmax_subset) & (vmax_subset > 0)
    idx = np.where(mask0)[0][good]
    if len(idx) < 50:
        print(f'{label}: not enough galaxies after cuts ({len(idx)})')
        return None

    weights = comp_weight[idx] if use_comp_weight else np.ones(len(idx), dtype=float)
    fit = fit_schechter_logL_vmax(logL[idx], vmax_subset[good], weights=weights, n_bins=N_DIAG_BINS)
    fit['label'] = label
    fit['z_range'] = z_range
    fit['mag_limit'] = mag_limit
    fit['idx'] = idx
    fit['use_comp_weight'] = use_comp_weight
    fit['z_values'] = z[idx]
    fit['M_star_minus_5logh'] = M_minus_5logh_from_logL(fit['logL_star'])
    param_err = parameter_errors_from_covariance(fit['cov'])
    fit['log_phi_star_err'] = param_err[0]
    fit['logL_star_err'] = param_err[1]
    fit['alpha_err'] = param_err[2]
    fit['M_star_minus_5logh_err'] = 2.5 * fit['logL_star_err']

    y = np.asarray(fit['density'], dtype=float)
    yerr = np.asarray(fit['density_err_eff'], dtype=float)
    x = np.asarray(fit['density_centers'], dtype=float)
    model = schechter_logL(x, fit['log_phi_star'], fit['logL_star'], fit['alpha'])
    pull = (y - model) / yerr
    chi2 = np.sum(pull**2)
    dof = len(y) - 3
    fit['chi2'] = chi2
    fit['dof'] = dof
    fit['reduced_chi2'] = chi2 / dof if dof > 0 else np.nan
    fit['p_value'] = chi2_dist.sf(chi2, dof) if dof > 0 else np.nan
    fit['pull'] = pull
    return fit


def summarize_fits(fits):
    rows = []
    for fit in fits:
        if fit is None:
            continue
        rows.append({
            'label': fit['label'],
            'z_low': fit['z_range'][0],
            'z_high': fit['z_range'][1],
            'r_mag_limit': fit['mag_limit'],
            'N': fit['N'],
            'weighted_N': fit.get('weight_sum_total', np.nan),
            'median_COMP_WEIGHT': fit.get('median_weight', np.nan),
            'use_comp_weight': fit.get('use_comp_weight', True),
            'log10_L_star': fit['logL_star'],
            'log10_L_star_err': fit.get('logL_star_err', np.nan),
            'M_star_minus_5logh': fit['M_star_minus_5logh'],
            'M_star_minus_5logh_err': fit.get('M_star_minus_5logh_err', np.nan),
            'alpha': fit['alpha'],
            'alpha_err': fit.get('alpha_err', np.nan),
            'phi_star': np.exp(fit['log_phi_star']),
            'chi2': fit['chi2'],
            'dof': fit['dof'],
            'reduced_chi2': fit['reduced_chi2'],
            'p_value': fit['p_value'],
        })
    return pd.DataFrame(rows)


def plot_lf_fit_comparison(fits, title=None, xlim=None, ylim=None):
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(fits), 1)))
    for fit, color in zip(fits, colors):
        if fit is None:
            continue
        x = fit['density_centers']
        y = fit['density']
        yerr = fit['density_err_eff']
        xerr = fit['density_xerr']
        grid = np.linspace(np.nanmin(x) - 0.05, np.nanmax(x) + 0.05, 400)
        model = schechter_logL(grid, fit['log_phi_star'], fit['logL_star'], fit['alpha'])
        ax.errorbar(x, y, xerr=xerr, yerr=yerr, fmt='o', ms=3.8, color=color, ecolor=color, alpha=0.75, capsize=0)
        ax.plot(grid, model, color=color, lw=2.0, label=rf"{fit['label']}: $\alpha={fit['alpha']:.2f}\pm{fit.get('alpha_err', np.nan):.2f}$, $M_*-5\log h={fit['M_star_minus_5logh']:.2f}\pm{fit.get('M_star_minus_5logh_err', np.nan):.2f}$")
    ax.axvline(LOG_L_MIN_FIT, color='0.35', ls='-.', lw=1.2)
    ax.set_yscale('log')
    ax.set_xlabel(r'$\log_{10}L_h$')
    ax.set_ylabel(r'$\phi(\log_{10}L)$ [$h^3\,{\rm Mpc}^{-3}{\rm dex}^{-1}$]')
    if title:
        ax.set_title(title)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(loc='best', fontsize=8)
    ax.tick_params(direction='in', top=True, right=True)
    fig.tight_layout()
    plt.show()

### Residuals colored by redshift

For each luminosity bin, we compute the mean redshift of galaxies contributing to that bin and color the residual pull by this mean redshift. A luminosity-dependent residual that is also redshift-ordered is a strong sign of redshift-dependent selection, K-correction, or evolution effects.

In [ ]:
def mean_z_per_fit_bin(fit):
    idx = fit.get('idx', None)
    if idx is None:
        raise KeyError("This fit dictionary has no 'idx'. Re-run the main global-fit cell after the notebook update so global_fit stores its row indices.")
    idx = np.asarray(idx, dtype=int)
    edges = fit['edges']
    centers = fit['density_centers']
    mean_z = np.full(len(centers), np.nan)
    bin_index = np.digitize(logL[idx], edges) - 1
    for k, center in enumerate(centers):
        # Match the selected good fit bins by nearest center.
        ibin = np.argmin(np.abs(fit['centers'] - center))
        in_bin = bin_index == ibin
        if np.any(in_bin):
            mean_z[k] = np.nanmean(z[idx][in_bin])
    return mean_z

mean_z_bin = mean_z_per_fit_bin(global_fit)

fig, ax = plt.subplots(figsize=(7.0, 4.6))
sc = ax.scatter(
    global_fit['density_centers'],
    pull,
    c=mean_z_bin,
    cmap='viridis',
    s=45,
    edgecolor='black',
    linewidth=0.4,
)
ax.axhline(0.0, color='black', lw=1.0)
ax.axvline(LOG_L_MIN_FIT, color='0.35', ls='-.', lw=1.2)
ax.set_xlabel(r'$\log_{10}L_h$')
ax.set_ylabel(r'pull')
ax.set_ylim(-5, 5)
ax.tick_params(direction='in', top=True, right=True)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r'mean $z$ in bin')
fig.tight_layout()
plt.show()


### NGC versus SGC

This checks whether the residual trend changes between the two Galactic caps. A strong NGC/SGC difference suggests footprint, calibration, target-selection, or cosmic-variance effects.

In [ ]:
fit_ngc = fit_lf_subset('NGC', extra_mask=(np.asarray(bgs['CAP']) == 'NGC'))
fit_sgc = fit_lf_subset('SGC', extra_mask=(np.asarray(bgs['CAP']) == 'SGC'))
fits_cap = [fit_ngc, fit_sgc]
display(summarize_fits(fits_cap))
plot_lf_fit_comparison(fits_cap, title='NGC versus SGC')

### Impact of `PROB_OBS` weighting

The DR2 catalog provides `PROB_OBS`, so the fiducial LF uses $w_{\rm completeness}=1/{\rm PROB\_OBS}$. This diagnostic compares the fiducial weighted LF to an unweighted fit. Differences here quantify how strongly the spectroscopic observation probability affects the inferred LF shape.


In [ ]:
fit_weighted = fit_lf_subset('weighted by 1/PROB_OBS', use_comp_weight=True)
fit_unweighted = fit_lf_subset('unweighted', use_comp_weight=False)
fits_prob_obs = [fit_weighted, fit_unweighted]
display(summarize_fits(fits_prob_obs))
plot_lf_fit_comparison(fits_prob_obs, title='Impact of PROB_OBS weighting')


### Red versus blue galaxies

This is a simple observed-frame color split using $g-r$. Because the color is not necessarily rest-frame corrected, treat this as a diagnostic. A strong difference is expected: red and blue populations are often better represented by separate Schechter components.

In [ ]:
color_good = np.isfinite(g_minus_r)
fit_blue = fit_lf_subset('blue', extra_mask=color_good & (g_minus_r < COLOR_SPLIT_GR))
fit_red = fit_lf_subset('red', extra_mask=color_good & (g_minus_r >= COLOR_SPLIT_GR))
fits_color = [fit_blue, fit_red]
display(summarize_fits(fits_color))
plot_lf_fit_comparison(fits_color, title=rf'Red versus blue split at $g-r={COLOR_SPLIT_GR:.2f}$')

### Apparent-magnitude cut checks within BGS Bright

The DR2 input used here is `BGS_BRIGHT_full.dat.fits`, so this notebook no longer compares BGS Bright against BGS Faint. Instead, we compare the fiducial BGS Bright limit, $r<19.5$, to a brighter internal cut. This checks whether the LF shape is sensitive to the flux limit while staying inside the same parent catalog.


In [ ]:
fit_r195 = fit_lf_subset(r'$r<19.5$', mag_limit=BGS_BRIGHT_LIMIT)
fit_r190 = fit_lf_subset(r'$r<19.0$', mag_limit=BRIGHTER_LIMIT)
fits_maglim = [fit_r195, fit_r190]
display(summarize_fits(fits_maglim))
plot_lf_fit_comparison(fits_maglim, title='BGS Bright magnitude-cut comparison')


### Narrow redshift range: $0.1<z<0.2$

A narrower redshift range reduces luminosity-redshift coupling, K-correction sensitivity, and LF evolution. If this fit looks much better, the global residual trend is likely driven by combining a broad flux-limited redshift interval.

In [ ]:
fit_z0102 = fit_lf_subset(r'$0.1<z<0.2$', z_range=(0.1, 0.2), mag_limit=BGS_BRIGHT_LIMIT)
fit_z0104 = fit_lf_subset(r'$0.1<z<0.4$', z_range=(0.1, 0.4), mag_limit=BGS_BRIGHT_LIMIT)
fits_zrange = [fit_z0102, fit_z0104]
display(summarize_fits(fits_zrange))
plot_lf_fit_comparison(fits_zrange, title='Narrow versus full redshift range')


## Save summary table

This writes only fit diagnostics, not a modified catalog. The production catalog-weighting script is `make_catalogs/fit_luminosity_function_weights.py`.

In [ ]:
summary_path = OUTPUT_DIR / 'bgs_direct_lf_logL_global_vmax_schechter_fit_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)